# 🔴 Solution: KV Cache

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

class KVCache:
    def __init__(self, num_heads: int, head_dim: int, max_seq_len: int, batch_size: int):
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        self.batch_size = batch_size
        self.seq_len = 0
        
        # Pre-allocate cache tensors
        self.k_cache = torch.zeros(batch_size, num_heads, max_seq_len, head_dim)
        self.v_cache = torch.zeros(batch_size, num_heads, max_seq_len, head_dim)
    
    def append(self, k: torch.Tensor, v: torch.Tensor) -> None:
        # k, v: (batch, num_heads, seq_len, head_dim)
        new_seq_len = k.shape[2]
        self.k_cache[:, :, self.seq_len:self.seq_len + new_seq_len] = k
        self.v_cache[:, :, self.seq_len:self.seq_len + new_seq_len] = v
        self.seq_len += new_seq_len
    
    def get(self) -> tuple[torch.Tensor, torch.Tensor]:
        return (
            self.k_cache[:, :, :self.seq_len],
            self.v_cache[:, :, :self.seq_len]
        )
    
    def clear(self) -> None:
        self.seq_len = 0
        self.k_cache.zero_()
        self.v_cache.zero_()

In [ ]:
# Verify
cache = KVCache(num_heads=8, head_dim=64, max_seq_len=256, batch_size=2)
k1 = torch.randn(2, 8, 3, 64)
v1 = torch.randn(2, 8, 3, 64)
cache.append(k1, v1)
k, v = cache.get()
print(f"K shape: {k.shape}, seq_len: {cache.seq_len}")

k2 = torch.randn(2, 8, 2, 64)
v2 = torch.randn(2, 8, 2, 64)
cache.append(k2, v2)
k, v = cache.get()
print(f"K shape after second append: {k.shape}, seq_len: {cache.seq_len}")

In [ ]:
from torch_judge import check
check("kv_cache")